## 1. Khai báo thư viện và thiết lập đường dẫn

Notebook này thực hiện bước sinh luật kết hợp từ frequent itemsets đã tạo ở bước trước.

Nếu frequent itemsets cho biết các sản phẩm hoặc nhóm sản phẩm nào thường xuất hiện cùng nhau, thì association rules sẽ chuyển các nhóm sản phẩm đó thành các luật dạng:

`A -> B`

Ví dụ, từ frequent itemset `{Banana, Organic Strawberries}`, ta có thể sinh ra các luật:

- `Banana -> Organic Strawberries`
- `Organic Strawberries -> Banana`

Các luật này sẽ được đánh giá bằng các chỉ số như support, confidence, lift, leverage và conviction. Ngoài ra, notebook cũng bổ sung một chỉ số cải tiến là `weighted_recommendation_score` để phục vụ việc xếp hạng gợi ý sản phẩm trong web demo.

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import warnings

from mlxtend.frequent_patterns import association_rules

warnings.filterwarnings("ignore")

cwd = Path.cwd()

if cwd.name.lower() == "notebooks":
    PROJECT_DIR = cwd.parent
else:
    PROJECT_DIR = cwd

PROCESSED_DIR = PROJECT_DIR / "data" / "processed"
RESULTS_DIR = PROJECT_DIR / "results"

RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print("PROJECT_DIR:", PROJECT_DIR)
print("PROCESSED_DIR:", PROCESSED_DIR)
print("RESULTS_DIR:", RESULTS_DIR)

PROJECT_DIR: d:\HK2_NAM3\KTDL_KPTT\CUOI_KY
PROCESSED_DIR: d:\HK2_NAM3\KTDL_KPTT\CUOI_KY\data\processed
RESULTS_DIR: d:\HK2_NAM3\KTDL_KPTT\CUOI_KY\results


## 2. Kiểm tra các file đầu vào cần thiết

Trước khi sinh luật kết hợp, notebook cần kiểm tra các file đầu vào đã được tạo đúng từ các bước trước hay chưa.

Các file cần thiết gồm:

| File | Vai trò |
| --- | --- |
| `frequent_itemsets.pkl` | Frequent itemsets được tạo từ FP-Growth, giữ nguyên kiểu dữ liệu `frozenset` |
| `product_mapping_sample.csv` | Ánh xạ product_id sang tên sản phẩm, quầy hàng và nhóm hàng |
| `basket_matrix_sparse.pkl` | Ma trận giao dịch - sản phẩm, dùng để lấy tổng số giao dịch |

Trong đó, `frequent_itemsets.pkl` là file quan trọng nhất cho bước sinh association rules.

In [2]:
required_files = {
    "frequent_itemsets.pkl": RESULTS_DIR / "frequent_itemsets.pkl",
    "product_mapping_sample.csv": PROCESSED_DIR / "product_mapping_sample.csv",
    "basket_matrix_sparse.pkl": PROCESSED_DIR / "basket_matrix_sparse.pkl",
}

file_check_rows = []

for file_name, file_path in required_files.items():
    exists = file_path.exists()
    size_mb = file_path.stat().st_size / (1024 * 1024) if exists else 0
    
    file_check_rows.append({
        "file_name": file_name,
        "path": str(file_path),
        "exists": exists,
        "size_mb": round(size_mb, 2)
    })

file_check_df = pd.DataFrame(file_check_rows)
display(file_check_df)

missing_files = file_check_df[file_check_df["exists"] == False]["file_name"].tolist()

if len(missing_files) == 0:
    print("Trạng thái: Tất cả file đầu vào cần thiết đều tồn tại.")
else:
    print("Trạng thái: Thiếu file đầu vào.")
    print(missing_files)

,file_name,path,exists,size_mb
0,frequent_itemsets.pkl,d:\HK2_NAM3\KTDL_KPTT\CUOI_KY\results\frequent...,True,0.30
1,product_mapping_sample.csv,d:\HK2_NAM3\KTDL_KPTT\CUOI_KY\data\processed\p...,True,0.17
2,basket_matrix_sparse.pkl,d:\HK2_NAM3\KTDL_KPTT\CUOI_KY\data\processed\b...,True,3.35


Trạng thái: Tất cả file đầu vào cần thiết đều tồn tại.


## 3. Đọc frequent itemsets, product mapping và basket matrix

Bước này đọc lại kết quả frequent itemsets đã tạo ở notebook trước. File `.pkl` được sử dụng vì nó giữ nguyên kiểu dữ liệu `frozenset` trong cột `itemsets`, phù hợp với hàm `association_rules()` của thư viện `mlxtend`.

Notebook cũng đọc `product_mapping_sample.csv` để chuyển mã sản phẩm sang tên sản phẩm, quầy hàng và nhóm ngành hàng. Ngoài ra, `basket_matrix_sparse.pkl` được đọc để lấy tổng số giao dịch, phục vụ việc tính `support_count` cho phần cải tiến phương pháp.

In [3]:
frequent_itemsets = pd.read_pickle(RESULTS_DIR / "frequent_itemsets.pkl")
product_mapping = pd.read_csv(PROCESSED_DIR / "product_mapping_sample.csv")
basket_matrix = pd.read_pickle(PROCESSED_DIR / "basket_matrix_sparse.pkl")

num_transactions = basket_matrix.shape[0]

print("Trạng thái: Load dữ liệu thành công.")
print("Kích thước frequent_itemsets:", frequent_itemsets.shape)
print("Kích thước product_mapping:", product_mapping.shape)
print("Kích thước basket_matrix:", basket_matrix.shape)
print("Số giao dịch:", num_transactions)

display(frequent_itemsets.head(10))

Trạng thái: Load dữ liệu thành công.
Kích thước frequent_itemsets: (4269, 4)
Kích thước product_mapping: (3000, 4)
Kích thước basket_matrix: (75519, 3000)
Số giao dịch: 75519


,itemsets,itemset_names,support,length
56,frozenset({24852}),Banana,0.143884,1
16,frozenset({13176}),Bag of Organic Bananas,0.121837,1
62,frozenset({21137}),Organic Strawberries,0.087024,1
74,frozenset({21903}),Organic Baby Spinach,0.077874,1
0,frozenset({47209}),Organic Hass Avocado,0.069784,1
205,frozenset({47766}),Organic Avocado,0.059561,1
23,frozenset({26209}),Limes,0.049166,1
1,frozenset({47626}),Large Lemon,0.047789,1
199,frozenset({27845}),Organic Whole Milk,0.046465,1
89,frozenset({16797}),Strawberries,0.046372,1


## 4. Kiểm tra cấu trúc frequent itemsets

Trước khi sinh luật kết hợp, cần kiểm tra frequent itemsets có đủ hai cột quan trọng là `support` và `itemsets` hay không.

Trong đó:

| Cột | Ý nghĩa |
| --- | --- |
| `support` | Tỷ lệ giao dịch chứa itemset |
| `itemsets` | Tập sản phẩm phổ biến, thường được lưu dưới dạng `frozenset` |

Nếu cột `itemsets` không còn ở dạng `frozenset`, thường là do đọc từ CSV thay vì Pickle, khi đó sẽ không phù hợp để sinh association rules trực tiếp.

In [4]:
print("Các cột trong frequent_itemsets:")
print(frequent_itemsets.columns.tolist())

required_columns = {"support", "itemsets"}
missing_columns = required_columns - set(frequent_itemsets.columns)

if len(missing_columns) == 0:
    print("Trạng thái: frequent_itemsets có đủ cột cần thiết.")
else:
    raise ValueError(f"frequent_itemsets thiếu các cột: {missing_columns}")

sample_itemset = frequent_itemsets["itemsets"].iloc[0]

print("Kiểu dữ liệu mẫu của itemsets:", type(sample_itemset))
print("Itemset mẫu:", sample_itemset)

if not isinstance(sample_itemset, frozenset):
    raise TypeError("Cột itemsets không phải frozenset. Hãy đọc frequent_itemsets từ file .pkl thay vì .csv.")

Các cột trong frequent_itemsets:
['itemsets', 'itemset_names', 'support', 'length']
Trạng thái: frequent_itemsets có đủ cột cần thiết.
Kiểu dữ liệu mẫu của itemsets: <class 'frozenset'>
Itemset mẫu: frozenset({24852})


## 5. Tạo hàm ánh xạ mã sản phẩm sang thông tin dễ hiểu

Các thuật toán khai thác luật kết hợp xử lý sản phẩm bằng `product_id`. Tuy nhiên, để trình bày trong báo cáo và web demo, kết quả cần được chuyển sang tên sản phẩm dễ hiểu.

Bước này tạo các dictionary để ánh xạ:

- `product_id` sang `product_name`
- `product_id` sang `department_name`
- `product_id` sang `aisle_name`

Sau đó, các hàm hỗ trợ được xây dựng để chuyển itemset từ dạng mã số sang tên sản phẩm, nhóm ngành hàng và quầy hàng.

In [6]:
product_mapping["product_id"] = product_mapping["product_id"].astype(str)

product_id_to_name = dict(
    zip(
        product_mapping["product_id"],
        product_mapping["product_name"]
    )
)

product_id_to_department = dict(
    zip(
        product_mapping["product_id"],
        product_mapping["department_name"]
    )
)

product_id_to_aisle = dict(
    zip(
        product_mapping["product_id"],
        product_mapping["aisle_name"]
    )
)

def sort_itemset(itemset):
    return sorted(list(itemset), key=lambda x: str(x))

def itemset_to_ids(itemset):
    return ", ".join([str(item) for item in sort_itemset(itemset)])

def itemset_to_names(itemset):
    names = []
    for item in sort_itemset(itemset):
        product_name = product_id_to_name.get(str(item), str(item))
        names.append(product_name)
    return ", ".join(names)

def itemset_to_departments(itemset):
    departments = []
    for item in sort_itemset(itemset):
        department_name = product_id_to_department.get(str(item), "Unknown")
        departments.append(department_name)
    return ", ".join(sorted(set(departments)))

def itemset_to_aisles(itemset):
    aisles = []
    for item in sort_itemset(itemset):
        aisle_name = product_id_to_aisle.get(str(item), "Unknown")
        aisles.append(aisle_name)
    return ", ".join(sorted(set(aisles)))

print("Trạng thái: Đã tạo hàm ánh xạ product_id sang thông tin sản phẩm.")
print("Số sản phẩm trong mapping:", len(product_id_to_name))

Trạng thái: Đã tạo hàm ánh xạ product_id sang thông tin sản phẩm.
Số sản phẩm trong mapping: 3000


## 6. Sinh luật kết hợp từ frequent itemsets

Ở bước này, notebook sử dụng hàm `association_rules()` để sinh luật kết hợp từ frequent itemsets.

Mỗi luật có dạng:

`antecedents -> consequents`

Trong đó:

| Thành phần | Ý nghĩa |
| --- | --- |
| `antecedents` | Vế trái của luật |
| `consequents` | Vế phải của luật |
| `support` | Tỷ lệ giao dịch chứa cả hai vế |
| `confidence` | Khi vế trái xuất hiện, xác suất vế phải xuất hiện |
| `lift` | Mức độ liên hệ giữa hai vế so với ngẫu nhiên |

Ở bước sinh ban đầu, notebook dùng ngưỡng confidence thấp là `0.05` để giữ lại nhiều luật tiềm năng. Sau đó, các luật sẽ được lọc chặt hơn bằng nhiều tiêu chí.

In [8]:
min_confidence_for_generation = 0.05

try:
    rules = association_rules(
        frequent_itemsets,
        metric="confidence",
        min_threshold=min_confidence_for_generation
    )
except TypeError:
    rules = association_rules(
        frequent_itemsets,
        num_itemsets=num_transactions,
        metric="confidence",
        min_threshold=min_confidence_for_generation
    )

print("Trạng thái: Đã sinh luật kết hợp.")
print("Số luật ban đầu:", len(rules))

display(rules.head(10))

Trạng thái: Đã sinh luật kết hợp.
Số luật ban đầu: 3560


,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,representativity,leverage,conviction,zhangs_metric,jaccard,certainty,kulczynski
0,frozenset({13176}),frozenset({21137}),0.121837,0.087024,0.020816,0.170851,1.963253,1.0,0.010213,1.101099,0.558713,0.110696,0.091817,0.205024
1,frozenset({21137}),frozenset({13176}),0.087024,0.121837,0.020816,0.239197,1.963253,1.0,0.010213,1.154258,0.537409,0.110696,0.133642,0.205024
2,frozenset({13176}),frozenset({47209}),0.121837,0.069784,0.019598,0.160852,2.305007,1.0,0.011095,1.108525,0.644712,0.113925,0.097900,0.220843
3,frozenset({47209}),frozenset({13176}),0.069784,0.121837,0.019598,0.280835,2.305007,1.0,0.011095,1.221087,0.608635,0.113925,0.181058,0.220843
4,frozenset({13176}),frozenset({21903}),0.121837,0.077874,0.016552,0.135855,1.744536,1.0,0.007064,1.067096,0.485994,0.090370,0.062877,0.174202
5,frozenset({21903}),frozenset({13176}),0.077874,0.121837,0.016552,0.212549,1.744536,1.0,0.007064,1.115197,0.462824,0.090370,0.103297,0.174202
6,frozenset({24852}),frozenset({47766}),0.143884,0.059561,0.016208,0.112645,1.891248,1.0,0.007638,1.059822,0.550450,0.086563,0.056446,0.192383
7,frozenset({47766}),frozenset({24852}),0.059561,0.143884,0.016208,0.272121,1.891248,1.0,0.007638,1.176178,0.501094,0.086563,0.149789,0.192383
8,frozenset({24852}),frozenset({21903}),0.143884,0.077874,0.015731,0.109332,1.403950,1.0,0.004526,1.035319,0.336081,0.076355,0.034114,0.155669
9,frozenset({21903}),frozenset({24852}),0.077874,0.143884,0.015731,0.202006,1.403950,1.0,0.004526,1.072835,0.312023,0.076355,0.067891,0.155669


## 7. Bổ sung tên sản phẩm, nhóm hàng và quầy hàng cho luật

Các luật kết hợp ban đầu vẫn còn ở dạng mã sản phẩm. Bước này bổ sung thêm các cột dễ hiểu hơn cho cả vế trái và vế phải của luật.

Các thông tin được bổ sung gồm:

| Thông tin | Ý nghĩa |
| --- | --- |
| `antecedent_names` | Tên sản phẩm ở vế trái |
| `consequent_names` | Tên sản phẩm ở vế phải |
| `antecedent_departments` | Nhóm hàng của vế trái |
| `consequent_departments` | Nhóm hàng của vế phải |
| `antecedent_aisles` | Quầy hàng của vế trái |
| `consequent_aisles` | Quầy hàng của vế phải |

Việc bổ sung các thông tin này giúp kết quả luật kết hợp có thể được phân tích theo dữ liệu có nhãn như department và aisle.

In [9]:
rules["antecedent_ids"] = rules["antecedents"].apply(itemset_to_ids)
rules["consequent_ids"] = rules["consequents"].apply(itemset_to_ids)

rules["antecedent_names"] = rules["antecedents"].apply(itemset_to_names)
rules["consequent_names"] = rules["consequents"].apply(itemset_to_names)

rules["antecedent_departments"] = rules["antecedents"].apply(itemset_to_departments)
rules["consequent_departments"] = rules["consequents"].apply(itemset_to_departments)

rules["antecedent_aisles"] = rules["antecedents"].apply(itemset_to_aisles)
rules["consequent_aisles"] = rules["consequents"].apply(itemset_to_aisles)

rules["antecedent_len"] = rules["antecedents"].apply(len)
rules["consequent_len"] = rules["consequents"].apply(len)
rules["rule_len"] = rules["antecedent_len"] + rules["consequent_len"]

print("Trạng thái: Đã bổ sung tên sản phẩm, nhóm hàng và quầy hàng.")
print("Số luật:", len(rules))

display(
    rules[
        [
            "antecedent_names",
            "consequent_names",
            "support",
            "confidence",
            "lift",
            "antecedent_departments",
            "consequent_departments",
            "rule_len"
        ]
    ].head(10)
)

Trạng thái: Đã bổ sung tên sản phẩm, nhóm hàng và quầy hàng.
Số luật: 3560


,antecedent_names,consequent_names,support,confidence,lift,antecedent_departments,consequent_departments,rule_len
0,Bag of Organic Bananas,Organic Strawberries,0.020816,0.170851,1.963253,produce,produce,2
1,Organic Strawberries,Bag of Organic Bananas,0.020816,0.239197,1.963253,produce,produce,2
2,Bag of Organic Bananas,Organic Hass Avocado,0.019598,0.160852,2.305007,produce,produce,2
3,Organic Hass Avocado,Bag of Organic Bananas,0.019598,0.280835,2.305007,produce,produce,2
4,Bag of Organic Bananas,Organic Baby Spinach,0.016552,0.135855,1.744536,produce,produce,2
5,Organic Baby Spinach,Bag of Organic Bananas,0.016552,0.212549,1.744536,produce,produce,2
6,Banana,Organic Avocado,0.016208,0.112645,1.891248,produce,produce,2
7,Organic Avocado,Banana,0.016208,0.272121,1.891248,produce,produce,2
8,Banana,Organic Baby Spinach,0.015731,0.109332,1.403950,produce,produce,2
9,Organic Baby Spinach,Banana,0.015731,0.202006,1.403950,produce,produce,2


## 8. Cải tiến phương pháp: xây dựng chỉ số xếp hạng gợi ý

Nếu chỉ sắp xếp luật theo confidence, kết quả có thể bị ảnh hưởng bởi các sản phẩm quá phổ biến. Nếu chỉ sắp xếp theo lift, một số luật có lift rất cao nhưng support thấp có thể được ưu tiên quá mức.

Vì vậy, notebook xây dựng hai chỉ số xếp hạng:

| Chỉ số | Công thức | Ý nghĩa |
| --- | --- | --- |
| `recommendation_score` | `confidence * lift` | Kết hợp khả năng mua kèm và mức độ liên hệ |
| `weighted_recommendation_score` | `confidence * lift * log(1 + support_count)` | Cân bằng thêm yếu tố số lần xuất hiện thực tế |

Trong đó:

`support_count = support * số lượng giao dịch`

Chỉ số `weighted_recommendation_score` là phần cải tiến phương pháp nhằm giúp việc xếp hạng gợi ý sản phẩm cân bằng hơn giữa độ phổ biến, khả năng mua kèm và mức độ liên hệ thực sự.

In [19]:
rules["support_count"] = rules["support"] * num_transactions

rules["recommendation_score"] = rules["confidence"] * rules["lift"]

rules["weighted_recommendation_score"] = (
    rules["confidence"] *
    rules["lift"] *
    np.log1p(rules["support_count"])
)

score_columns = [
    "antecedent_names",
    "consequent_names",
    "support",
    "support_count",
    "confidence",
    "lift",
    "recommendation_score",
    "weighted_recommendation_score"
]

print("Trạng thái: Đã tạo recommendation_score và weighted_recommendation_score.")

display(
    rules[score_columns]
    .sort_values("weighted_recommendation_score", ascending=False)
    .head(10)
)

Trạng thái: Đã tạo recommendation_score và weighted_recommendation_score.


,antecedent_names,consequent_names,support,support_count,confidence,lift,recommendation_score,weighted_recommendation_score
2874,Peach on the Bottom Nonfat Greek Yogurt,Blueberry on the Bottom Nonfat Greek Yogurt,0.001126,85.0,0.384615,123.075293,47.336651,210.853885
2873,Blueberry on the Bottom Nonfat Greek Yogurt,Peach on the Bottom Nonfat Greek Yogurt,0.001126,85.0,0.360169,123.075293,44.327966,197.452155
1824,Zero Calorie Cola,Soda,0.001417,107.0,0.543147,55.131632,29.944592,140.204509
1599,Icelandic Style Skyr Blueberry Non-fat Yogurt,Vanilla Skyr Nonfat Yogurt,0.001523,115.0,0.344311,67.713674,23.314588,110.827999
3314,"Sparkling Water Berry, Lime Sparkling Water",Sparkling Lemon Water,0.001033,78.0,0.520000,48.361921,25.148199,109.883744
818,"Sparkling Lemon Water, Lime Sparkling Water",Sparkling Water Grapefruit,0.002172,164.0,0.735426,28.120829,20.680789,105.594980
2480,"Sparkling Water Berry, Sparkling Water Grapefruit",Sparkling Lemon Water,0.001218,92.0,0.491979,45.755828,22.510889,102.032843
1684,"Sparkling Water Berry, Sparkling Water Grapefruit",Lime Sparkling Water,0.001470,111.0,0.593583,36.296993,21.545274,101.661350
1685,"Sparkling Water Berry, Lime Sparkling Water",Sparkling Water Grapefruit,0.001470,111.0,0.740000,28.295727,20.938838,98.799882
3313,"Sparkling Water Berry, Sparkling Lemon Water",Lime Sparkling Water,0.001033,78.0,0.604651,36.973807,22.356255,97.684491


## 9. Chuẩn hóa bảng luật để phục vụ phân tích và xuất file

Bước này chọn ra các cột quan trọng và tạo bảng `rules_enriched`. Đây là bảng luật đầy đủ đã được làm giàu thông tin, bao gồm:

- Mã sản phẩm
- Tên sản phẩm
- Các chỉ số đánh giá luật
- Độ dài luật
- Nhóm hàng và quầy hàng
- Chỉ số xếp hạng gợi ý cải tiến

Bảng này sẽ là cơ sở để lọc luật, tạo top rules và xuất file kết quả.

In [20]:
rules_display_cols = [
    "antecedent_ids",
    "consequent_ids",
    "antecedent_names",
    "consequent_names",
    "support",
    "support_count",
    "confidence",
    "lift",
    "leverage",
    "conviction",
    "recommendation_score",
    "weighted_recommendation_score",
    "antecedent_len",
    "consequent_len",
    "rule_len",
    "antecedent_departments",
    "consequent_departments",
    "antecedent_aisles",
    "consequent_aisles"
]

available_cols = [col for col in rules_display_cols if col in rules.columns]

rules_enriched = rules[available_cols].copy()

print("Trạng thái: Đã tạo bảng rules_enriched.")
print("Số luật trong rules_enriched:", len(rules_enriched))

display(
    rules_enriched.sort_values(
        ["weighted_recommendation_score", "confidence", "lift"],
        ascending=False
    ).head(20)
)

Trạng thái: Đã tạo bảng rules_enriched.
Số luật trong rules_enriched: 3560


,antecedent_ids,consequent_ids,antecedent_names,consequent_names,support,support_count,confidence,lift,leverage,conviction,recommendation_score,weighted_recommendation_score,antecedent_len,consequent_len,rule_len,antecedent_departments,consequent_departments,antecedent_aisles,consequent_aisles
2874,33548,23296,Peach on the Bottom Nonfat Greek Yogurt,Blueberry on the Bottom Nonfat Greek Yogurt,0.001126,85.0,0.384615,123.075293,0.001116,1.619922,47.336651,210.853885,1,1,2,dairy eggs,dairy eggs,yogurt,yogurt
2873,23296,33548,Blueberry on the Bottom Nonfat Greek Yogurt,Peach on the Bottom Nonfat Greek Yogurt,0.001126,85.0,0.360169,123.075293,0.001116,1.558340,44.327966,197.452155,1,1,2,dairy eggs,dairy eggs,yogurt,yogurt
1824,46149,196,Zero Calorie Cola,Soda,0.001417,107.0,0.543147,55.131632,0.001391,2.167324,29.944592,140.204509,1,1,2,beverages,beverages,soft drinks,soft drinks
1599,28465,24799,Icelandic Style Skyr Blueberry Non-fat Yogurt,Vanilla Skyr Nonfat Yogurt,0.001523,115.0,0.344311,67.713674,0.001500,1.517359,23.314588,110.827999,1,1,2,dairy eggs,dairy eggs,yogurt,yogurt
3314,"20119, 35221",21709,"Sparkling Water Berry, Lime Sparkling Water",Sparkling Lemon Water,0.001033,78.0,0.520000,48.361921,0.001011,2.060933,25.148199,109.883744,2,1,3,beverages,beverages,water seltzer sparkling water,water seltzer sparkling water
818,"21709, 35221",44632,"Sparkling Lemon Water, Lime Sparkling Water",Sparkling Water Grapefruit,0.002172,164.0,0.735426,28.120829,0.002094,3.680814,20.680789,105.594980,2,1,3,beverages,beverages,water seltzer sparkling water,water seltzer sparkling water
2480,"20119, 44632",21709,"Sparkling Water Berry, Sparkling Water Grapefruit",Sparkling Lemon Water,0.001218,92.0,0.491979,45.755828,0.001192,1.947256,22.510889,102.032843,2,1,3,beverages,beverages,water seltzer sparkling water,water seltzer sparkling water
1684,"20119, 44632",35221,"Sparkling Water Berry, Sparkling Water Grapefruit",Lime Sparkling Water,0.001470,111.0,0.593583,36.296993,0.001429,2.420288,21.545274,101.661350,2,1,3,beverages,beverages,water seltzer sparkling water,water seltzer sparkling water
1685,"20119, 35221",44632,"Sparkling Water Berry, Lime Sparkling Water",Sparkling Water Grapefruit,0.001470,111.0,0.740000,28.295727,0.001418,3.745568,20.938838,98.799882,2,1,3,beverages,beverages,water seltzer sparkling water,water seltzer sparkling water
3313,"20119, 21709",35221,"Sparkling Water Berry, Sparkling Lemon Water",Lime Sparkling Water,0.001033,78.0,0.604651,36.973807,0.001005,2.488047,22.356255,97.684491,2,1,3,beverages,beverages,water seltzer sparkling water,water seltzer sparkling water


## 10. Lọc các luật kết hợp có chất lượng tốt

Không phải mọi luật được sinh ra đều có giá trị phân tích hoặc ứng dụng. Vì vậy, notebook lọc lại các luật dựa trên nhiều tiêu chí thay vì chỉ dùng một chỉ số đơn lẻ.

Các điều kiện lọc gồm:

| Điều kiện | Ý nghĩa |
| --- | --- |
| `support >= 0.001` | Luật phải xuất hiện đủ nhiều trong dữ liệu |
| `confidence >= 0.20` | Khi vế trái xuất hiện, vế phải có xác suất xuất hiện đủ cao |
| `lift >= 1.20` | Quan hệ giữa hai vế phải mạnh hơn mức ngẫu nhiên |
| `rule_len <= 4` | Giữ luật ở mức vừa phải, dễ giải thích |

Đây là một phần cải tiến phương pháp vì luật được chọn dựa trên nhiều tiêu chí cân bằng hơn, thay vì chỉ chọn theo confidence.

In [21]:
min_support_rule = 0.001
min_confidence_rule = 0.20
min_lift_rule = 1.20
max_rule_len = 4

rules_filtered = rules_enriched[
    (rules_enriched["support"] >= min_support_rule) &
    (rules_enriched["confidence"] >= min_confidence_rule) &
    (rules_enriched["lift"] >= min_lift_rule) &
    (rules_enriched["rule_len"] <= max_rule_len)
].copy()

rules_filtered = rules_filtered.sort_values(
    ["weighted_recommendation_score", "confidence", "lift", "support"],
    ascending=False
)

print("Ngưỡng lọc luật:")
print("min_support:", min_support_rule)
print("min_confidence:", min_confidence_rule)
print("min_lift:", min_lift_rule)
print("max_rule_len:", max_rule_len)
print("Số luật sau lọc:", len(rules_filtered))

display(rules_filtered.head(20))

Ngưỡng lọc luật:
min_support: 0.001
min_confidence: 0.2
min_lift: 1.2
max_rule_len: 4
Số luật sau lọc: 896


,antecedent_ids,consequent_ids,antecedent_names,consequent_names,support,support_count,confidence,lift,leverage,conviction,recommendation_score,weighted_recommendation_score,antecedent_len,consequent_len,rule_len,antecedent_departments,consequent_departments,antecedent_aisles,consequent_aisles
2874,33548,23296,Peach on the Bottom Nonfat Greek Yogurt,Blueberry on the Bottom Nonfat Greek Yogurt,0.001126,85.0,0.384615,123.075293,0.001116,1.619922,47.336651,210.853885,1,1,2,dairy eggs,dairy eggs,yogurt,yogurt
2873,23296,33548,Blueberry on the Bottom Nonfat Greek Yogurt,Peach on the Bottom Nonfat Greek Yogurt,0.001126,85.0,0.360169,123.075293,0.001116,1.558340,44.327966,197.452155,1,1,2,dairy eggs,dairy eggs,yogurt,yogurt
1824,46149,196,Zero Calorie Cola,Soda,0.001417,107.0,0.543147,55.131632,0.001391,2.167324,29.944592,140.204509,1,1,2,beverages,beverages,soft drinks,soft drinks
1599,28465,24799,Icelandic Style Skyr Blueberry Non-fat Yogurt,Vanilla Skyr Nonfat Yogurt,0.001523,115.0,0.344311,67.713674,0.001500,1.517359,23.314588,110.827999,1,1,2,dairy eggs,dairy eggs,yogurt,yogurt
3314,"20119, 35221",21709,"Sparkling Water Berry, Lime Sparkling Water",Sparkling Lemon Water,0.001033,78.0,0.520000,48.361921,0.001011,2.060933,25.148199,109.883744,2,1,3,beverages,beverages,water seltzer sparkling water,water seltzer sparkling water
818,"21709, 35221",44632,"Sparkling Lemon Water, Lime Sparkling Water",Sparkling Water Grapefruit,0.002172,164.0,0.735426,28.120829,0.002094,3.680814,20.680789,105.594980,2,1,3,beverages,beverages,water seltzer sparkling water,water seltzer sparkling water
2480,"20119, 44632",21709,"Sparkling Water Berry, Sparkling Water Grapefruit",Sparkling Lemon Water,0.001218,92.0,0.491979,45.755828,0.001192,1.947256,22.510889,102.032843,2,1,3,beverages,beverages,water seltzer sparkling water,water seltzer sparkling water
1684,"20119, 44632",35221,"Sparkling Water Berry, Sparkling Water Grapefruit",Lime Sparkling Water,0.001470,111.0,0.593583,36.296993,0.001429,2.420288,21.545274,101.661350,2,1,3,beverages,beverages,water seltzer sparkling water,water seltzer sparkling water
1685,"20119, 35221",44632,"Sparkling Water Berry, Lime Sparkling Water",Sparkling Water Grapefruit,0.001470,111.0,0.740000,28.295727,0.001418,3.745568,20.938838,98.799882,2,1,3,beverages,beverages,water seltzer sparkling water,water seltzer sparkling water
3313,"20119, 21709",35221,"Sparkling Water Berry, Sparkling Lemon Water",Lime Sparkling Water,0.001033,78.0,0.604651,36.973807,0.001005,2.488047,22.356255,97.684491,2,1,3,beverages,beverages,water seltzer sparkling water,water seltzer sparkling water


## 11. Tạo các bảng top luật theo từng tiêu chí

Sau khi lọc luật, notebook tạo các bảng top luật theo nhiều tiêu chí khác nhau.

| Bảng | Ý nghĩa |
| --- | --- |
| Top luật theo lift | Tìm các luật có mức liên hệ mạnh hơn ngẫu nhiên cao nhất |
| Top luật theo confidence | Tìm các luật có khả năng dự đoán vế phải tốt nhất |
| Top luật theo recommendation_score | Xếp hạng theo confidence và lift |
| Top luật theo weighted_recommendation_score | Xếp hạng cải tiến có xét thêm support_count |

Việc tạo nhiều bảng top rules giúp phần phân tích kết quả thực nghiệm sâu hơn, thay vì chỉ nhìn vào một chỉ số duy nhất.

In [13]:
top_rules_by_lift = rules_filtered.sort_values(
    ["lift", "confidence", "support"],
    ascending=False
).head(50)

top_rules_by_confidence = rules_filtered.sort_values(
    ["confidence", "lift", "support"],
    ascending=False
).head(50)

top_rules_by_score = rules_filtered.sort_values(
    ["recommendation_score", "confidence", "lift"],
    ascending=False
).head(50)

top_rules_by_weighted_score = rules_filtered.sort_values(
    ["weighted_recommendation_score", "recommendation_score", "confidence"],
    ascending=False
).head(50)

print("Top luật theo lift:")
display(top_rules_by_lift.head(10))

print("Top luật theo confidence:")
display(top_rules_by_confidence.head(10))

print("Top luật theo recommendation_score:")
display(top_rules_by_score.head(10))

print("Top luật theo weighted_recommendation_score:")
display(top_rules_by_weighted_score.head(10))

Top luật theo lift:


,antecedent_ids,consequent_ids,antecedent_names,consequent_names,support,support_count,confidence,lift,leverage,conviction,recommendation_score,weighted_recommendation_score,antecedent_len,consequent_len,rule_len,antecedent_departments,consequent_departments,antecedent_aisles,consequent_aisles
2874,33548,23296,Peach on the Bottom Nonfat Greek Yogurt,Blueberry on the Bottom Nonfat Greek Yogurt,0.001126,85.0,0.384615,123.075293,0.001116,1.619922,47.336651,210.853885,1,1,2,dairy eggs,dairy eggs,yogurt,yogurt
2873,23296,33548,Blueberry on the Bottom Nonfat Greek Yogurt,Peach on the Bottom Nonfat Greek Yogurt,0.001126,85.0,0.360169,123.075293,0.001116,1.558340,44.327966,197.452155,1,1,2,dairy eggs,dairy eggs,yogurt,yogurt
1599,28465,24799,Icelandic Style Skyr Blueberry Non-fat Yogurt,Vanilla Skyr Nonfat Yogurt,0.001523,115.0,0.344311,67.713674,0.001500,1.517359,23.314588,110.827999,1,1,2,dairy eggs,dairy eggs,yogurt,yogurt
1600,24799,28465,Vanilla Skyr Nonfat Yogurt,Icelandic Style Skyr Blueberry Non-fat Yogurt,0.001523,115.0,0.299479,67.713674,0.001500,1.421196,20.278835,96.397270,1,1,2,dairy eggs,dairy eggs,yogurt,yogurt
2864,12576,39947,Kiwi Sandia Sparkling Water,Blackberry Cucumber Sparkling Water,0.001126,85.0,0.272436,65.522569,0.001108,1.368735,17.850700,79.513216,1,1,2,beverages,beverages,water seltzer sparkling water,water seltzer sparkling water
2865,39947,12576,Blackberry Cucumber Sparkling Water,Kiwi Sandia Sparkling Water,0.001126,85.0,0.270701,65.522569,0.001108,1.365514,17.737001,79.006763,1,1,2,beverages,beverages,water seltzer sparkling water,water seltzer sparkling water
1586,4957,40571,Total 2% Lowfat Greek Strained Yogurt With Blu...,Total 2% Greek Strained Yogurt with Cherry 5.3 oz,0.001523,115.0,0.316804,64.313850,0.001499,1.456500,20.374911,96.853977,1,1,2,dairy eggs,dairy eggs,yogurt,yogurt
1585,40571,4957,Total 2% Greek Strained Yogurt with Cherry 5.3 oz,Total 2% Lowfat Greek Strained Yogurt With Blu...,0.001523,115.0,0.309140,64.313850,0.001499,1.440513,19.881970,94.510736,1,1,2,dairy eggs,dairy eggs,yogurt,yogurt
2143,36865,24799,Non Fat Raspberry Yogurt,Vanilla Skyr Nonfat Yogurt,0.001311,99.0,0.326733,64.256575,0.001291,1.477742,20.994722,96.684270,1,1,2,dairy eggs,dairy eggs,yogurt,yogurt
2144,24799,36865,Vanilla Skyr Nonfat Yogurt,Non Fat Raspberry Yogurt,0.001311,99.0,0.257812,64.256575,0.001291,1.341962,16.566148,76.289932,1,1,2,dairy eggs,dairy eggs,yogurt,yogurt


Top luật theo confidence:


,antecedent_ids,consequent_ids,antecedent_names,consequent_names,support,support_count,confidence,lift,leverage,conviction,recommendation_score,weighted_recommendation_score,antecedent_len,consequent_len,rule_len,antecedent_departments,consequent_departments,antecedent_aisles,consequent_aisles
1685,"20119, 35221",44632,"Sparkling Water Berry, Lime Sparkling Water",Sparkling Water Grapefruit,0.001470,111.0,0.740000,28.295727,0.001418,3.745568,20.938838,98.799882,2,1,3,beverages,beverages,water seltzer sparkling water,water seltzer sparkling water
818,"21709, 35221",44632,"Sparkling Lemon Water, Lime Sparkling Water",Sparkling Water Grapefruit,0.002172,164.0,0.735426,28.120829,0.002094,3.680814,20.680789,105.594980,2,1,3,beverages,beverages,water seltzer sparkling water,water seltzer sparkling water
2481,"20119, 21709",44632,"Sparkling Water Berry, Sparkling Lemon Water",Sparkling Water Grapefruit,0.001218,92.0,0.713178,27.270132,0.001174,3.395307,19.448467,88.152110,2,1,3,beverages,beverages,water seltzer sparkling water,water seltzer sparkling water
3464,43654,13176,Whole Milk Greek Blended Vanilla Bean Yogurt,Bag of Organic Bananas,0.001006,76.0,0.608000,4.990278,0.000805,2.240212,3.034089,13.179494,1,1,2,dairy eggs,produce,yogurt,fresh fruits
3313,"20119, 21709",35221,"Sparkling Water Berry, Sparkling Lemon Water",Lime Sparkling Water,0.001033,78.0,0.604651,36.973807,0.001005,2.488047,22.356255,97.684491,2,1,3,beverages,beverages,water seltzer sparkling water,water seltzer sparkling water
1684,"20119, 44632",35221,"Sparkling Water Berry, Sparkling Water Grapefruit",Lime Sparkling Water,0.001470,111.0,0.593583,36.296993,0.001429,2.420288,21.545274,101.661350,2,1,3,beverages,beverages,water seltzer sparkling water,water seltzer sparkling water
2552,"16797, 27845",24852,"Strawberries, Organic Whole Milk",Banana,0.001205,91.0,0.565217,3.928276,0.000898,1.969066,2.220330,10.039864,2,1,3,"dairy eggs, produce",produce,"fresh fruits, milk",fresh fruits
2893,"4920, 49683",24852,"Seedless Red Grapes, Cucumber Kirby",Banana,0.001126,85.0,0.555556,3.861126,0.000834,1.926260,2.145070,9.554888,2,1,3,produce,produce,"fresh vegetables, packaged vegetables fruits",fresh fruits
816,"21709, 44632",35221,"Sparkling Lemon Water, Sparkling Water Grapefruit",Lime Sparkling Water,0.002172,164.0,0.546667,33.428113,0.002107,2.169808,18.274035,93.306228,2,1,3,beverages,beverages,water seltzer sparkling water,water seltzer sparkling water
1824,46149,196,Zero Calorie Cola,Soda,0.001417,107.0,0.543147,55.131632,0.001391,2.167324,29.944592,140.204509,1,1,2,beverages,beverages,soft drinks,soft drinks


Top luật theo recommendation_score:


,antecedent_ids,consequent_ids,antecedent_names,consequent_names,support,support_count,confidence,lift,leverage,conviction,recommendation_score,weighted_recommendation_score,antecedent_len,consequent_len,rule_len,antecedent_departments,consequent_departments,antecedent_aisles,consequent_aisles
2874,33548,23296,Peach on the Bottom Nonfat Greek Yogurt,Blueberry on the Bottom Nonfat Greek Yogurt,0.001126,85.0,0.384615,123.075293,0.001116,1.619922,47.336651,210.853885,1,1,2,dairy eggs,dairy eggs,yogurt,yogurt
2873,23296,33548,Blueberry on the Bottom Nonfat Greek Yogurt,Peach on the Bottom Nonfat Greek Yogurt,0.001126,85.0,0.360169,123.075293,0.001116,1.558340,44.327966,197.452155,1,1,2,dairy eggs,dairy eggs,yogurt,yogurt
1824,46149,196,Zero Calorie Cola,Soda,0.001417,107.0,0.543147,55.131632,0.001391,2.167324,29.944592,140.204509,1,1,2,beverages,beverages,soft drinks,soft drinks
3314,"20119, 35221",21709,"Sparkling Water Berry, Lime Sparkling Water",Sparkling Lemon Water,0.001033,78.0,0.520000,48.361921,0.001011,2.060933,25.148199,109.883744,2,1,3,beverages,beverages,water seltzer sparkling water,water seltzer sparkling water
1599,28465,24799,Icelandic Style Skyr Blueberry Non-fat Yogurt,Vanilla Skyr Nonfat Yogurt,0.001523,115.0,0.344311,67.713674,0.001500,1.517359,23.314588,110.827999,1,1,2,dairy eggs,dairy eggs,yogurt,yogurt
2480,"20119, 44632",21709,"Sparkling Water Berry, Sparkling Water Grapefruit",Sparkling Lemon Water,0.001218,92.0,0.491979,45.755828,0.001192,1.947256,22.510889,102.032843,2,1,3,beverages,beverages,water seltzer sparkling water,water seltzer sparkling water
3313,"20119, 21709",35221,"Sparkling Water Berry, Sparkling Lemon Water",Lime Sparkling Water,0.001033,78.0,0.604651,36.973807,0.001005,2.488047,22.356255,97.684491,2,1,3,beverages,beverages,water seltzer sparkling water,water seltzer sparkling water
1684,"20119, 44632",35221,"Sparkling Water Berry, Sparkling Water Grapefruit",Lime Sparkling Water,0.001470,111.0,0.593583,36.296993,0.001429,2.420288,21.545274,101.661350,2,1,3,beverages,beverages,water seltzer sparkling water,water seltzer sparkling water
2143,36865,24799,Non Fat Raspberry Yogurt,Vanilla Skyr Nonfat Yogurt,0.001311,99.0,0.326733,64.256575,0.001291,1.477742,20.994722,96.684270,1,1,2,dairy eggs,dairy eggs,yogurt,yogurt
1685,"20119, 35221",44632,"Sparkling Water Berry, Lime Sparkling Water",Sparkling Water Grapefruit,0.001470,111.0,0.740000,28.295727,0.001418,3.745568,20.938838,98.799882,2,1,3,beverages,beverages,water seltzer sparkling water,water seltzer sparkling water


Top luật theo weighted_recommendation_score:


,antecedent_ids,consequent_ids,antecedent_names,consequent_names,support,support_count,confidence,lift,leverage,conviction,recommendation_score,weighted_recommendation_score,antecedent_len,consequent_len,rule_len,antecedent_departments,consequent_departments,antecedent_aisles,consequent_aisles
2874,33548,23296,Peach on the Bottom Nonfat Greek Yogurt,Blueberry on the Bottom Nonfat Greek Yogurt,0.001126,85.0,0.384615,123.075293,0.001116,1.619922,47.336651,210.853885,1,1,2,dairy eggs,dairy eggs,yogurt,yogurt
2873,23296,33548,Blueberry on the Bottom Nonfat Greek Yogurt,Peach on the Bottom Nonfat Greek Yogurt,0.001126,85.0,0.360169,123.075293,0.001116,1.558340,44.327966,197.452155,1,1,2,dairy eggs,dairy eggs,yogurt,yogurt
1824,46149,196,Zero Calorie Cola,Soda,0.001417,107.0,0.543147,55.131632,0.001391,2.167324,29.944592,140.204509,1,1,2,beverages,beverages,soft drinks,soft drinks
1599,28465,24799,Icelandic Style Skyr Blueberry Non-fat Yogurt,Vanilla Skyr Nonfat Yogurt,0.001523,115.0,0.344311,67.713674,0.001500,1.517359,23.314588,110.827999,1,1,2,dairy eggs,dairy eggs,yogurt,yogurt
3314,"20119, 35221",21709,"Sparkling Water Berry, Lime Sparkling Water",Sparkling Lemon Water,0.001033,78.0,0.520000,48.361921,0.001011,2.060933,25.148199,109.883744,2,1,3,beverages,beverages,water seltzer sparkling water,water seltzer sparkling water
818,"21709, 35221",44632,"Sparkling Lemon Water, Lime Sparkling Water",Sparkling Water Grapefruit,0.002172,164.0,0.735426,28.120829,0.002094,3.680814,20.680789,105.594980,2,1,3,beverages,beverages,water seltzer sparkling water,water seltzer sparkling water
2480,"20119, 44632",21709,"Sparkling Water Berry, Sparkling Water Grapefruit",Sparkling Lemon Water,0.001218,92.0,0.491979,45.755828,0.001192,1.947256,22.510889,102.032843,2,1,3,beverages,beverages,water seltzer sparkling water,water seltzer sparkling water
1684,"20119, 44632",35221,"Sparkling Water Berry, Sparkling Water Grapefruit",Lime Sparkling Water,0.001470,111.0,0.593583,36.296993,0.001429,2.420288,21.545274,101.661350,2,1,3,beverages,beverages,water seltzer sparkling water,water seltzer sparkling water
1685,"20119, 35221",44632,"Sparkling Water Berry, Lime Sparkling Water",Sparkling Water Grapefruit,0.001470,111.0,0.740000,28.295727,0.001418,3.745568,20.938838,98.799882,2,1,3,beverages,beverages,water seltzer sparkling water,water seltzer sparkling water
3313,"20119, 21709",35221,"Sparkling Water Berry, Sparkling Lemon Water",Lime Sparkling Water,0.001033,78.0,0.604651,36.973807,0.001005,2.488047,22.356255,97.684491,2,1,3,beverages,beverages,water seltzer sparkling water,water seltzer sparkling water


## 12. Tạo recommendation lookup cho web demo

Web demo cần một bảng đơn giản để khi người dùng chọn một sản phẩm, hệ thống có thể trả về các sản phẩm nên mua kèm.

Vì vậy, notebook tạo file `recommendation_lookup.csv` từ các luật có dạng:

`1 sản phẩm đầu vào -> 1 sản phẩm gợi ý`

Ví dụ:

`Banana -> Organic Strawberries`

Bảng này sẽ được web demo đọc trực tiếp để hiển thị gợi ý. Đây là cách triển khai hợp lý vì thuật toán khai thác luật được chạy offline bằng Python, còn web demo chỉ cần tra cứu kết quả đã lưu.

In [14]:
single_product_rules = rules_filtered[
    (rules_filtered["antecedent_len"] == 1) &
    (rules_filtered["consequent_len"] == 1)
].copy()

single_product_rules["input_product_id"] = single_product_rules["antecedent_ids"]
single_product_rules["input_product_name"] = single_product_rules["antecedent_names"]

single_product_rules["recommended_product_id"] = single_product_rules["consequent_ids"]
single_product_rules["recommended_product_name"] = single_product_rules["consequent_names"]

recommendation_columns = [
    "input_product_id",
    "input_product_name",
    "recommended_product_id",
    "recommended_product_name",
    "support",
    "support_count",
    "confidence",
    "lift",
    "leverage",
    "conviction",
    "recommendation_score",
    "weighted_recommendation_score",
    "antecedent_departments",
    "consequent_departments",
    "antecedent_aisles",
    "consequent_aisles"
]

available_recommendation_columns = [
    col for col in recommendation_columns
    if col in single_product_rules.columns
]

recommendation_lookup = single_product_rules[available_recommendation_columns].copy()

recommendation_lookup = recommendation_lookup.sort_values(
    ["input_product_name", "weighted_recommendation_score"],
    ascending=[True, False]
)

print("Số dòng trong recommendation_lookup:", len(recommendation_lookup))
print("Số sản phẩm đầu vào có thể gợi ý:", recommendation_lookup["input_product_name"].nunique())

display(recommendation_lookup.head(20))

Số dòng trong recommendation_lookup: 450
Số sản phẩm đầu vào có thể gợi ý: 306


,input_product_id,input_product_name,recommended_product_id,recommended_product_name,support,support_count,confidence,lift,leverage,conviction,recommendation_score,weighted_recommendation_score,antecedent_departments,consequent_departments,antecedent_aisles,consequent_aisles
3318,38928,0% Greek Strained Yogurt,6184,Clementines,0.001033,78.0,0.228739,26.332532,0.000994,1.285315,6.023277,26.318395,dairy eggs,produce,yogurt,packaged produce
3161,38928,0% Greek Strained Yogurt,13176,Bag of Organic Bananas,0.001059,80.0,0.234604,1.925559,0.000509,1.147332,0.451744,1.985166,dairy eggs,produce,yogurt,fresh fruits
1118,24024,1% Lowfat Milk,24852,Banana,0.001841,139.0,0.293249,2.038088,0.000937,1.211340,0.597667,2.953458,dairy eggs,produce,milk,fresh fruits
3149,15902,100 Calorie Per Bag Popcorn,24852,Banana,0.001059,80.0,0.287770,2.000008,0.000530,1.202021,0.575542,2.529189,snacks,produce,popcorn jerky,fresh fruits
590,3957,100% Raw Coconut Water,13176,Bag of Organic Bananas,0.002648,200.0,0.222965,1.830032,0.001201,1.130147,0.408034,2.163928,beverages,produce,refrigerated,fresh fruits
163,5077,100% Whole Wheat Bread,13176,Bag of Organic Bananas,0.004846,366.0,0.240473,1.973730,0.002391,1.156197,0.474629,2.802855,bakery,produce,bread,fresh fruits
2686,1511,2% Reduced Fat DHA Omega-3 Reduced Fat Milk,24852,Banana,0.001165,88.0,0.303448,2.108974,0.000613,1.229077,0.639964,2.872568,dairy eggs,produce,milk,fresh fruits
509,23909,2% Reduced Fat Milk,24852,Banana,0.002860,216.0,0.243792,1.694363,0.001172,1.132117,0.413073,2.222289,dairy eggs,produce,milk,fresh fruits
1095,40174,2% Reduced Fat Organic Milk,24852,Banana,0.001867,141.0,0.332547,2.311212,0.001059,1.282661,0.768587,3.808984,dairy eggs,produce,milk,fresh fruits
3096,38164,Almonds & Sea Salt in Dark Chocolate,13176,Bag of Organic Bananas,0.001073,81.0,0.200993,1.649686,0.000422,1.099067,0.331575,1.461156,snacks,produce,candy chocolate,fresh fruits


## 13. Tạo bảng tóm tắt kết quả sinh luật kết hợp

Bảng tóm tắt giúp kiểm soát kết quả của toàn bộ bước sinh luật kết hợp, gồm:

- Số frequent itemsets đầu vào
- Số luật ban đầu
- Số luật sau lọc
- Số luật dùng cho gợi ý một sản phẩm
- Các ngưỡng lọc đã sử dụng
- Tổng số giao dịch dùng để tính support_count

Bảng này có thể được đưa vào báo cáo để trình bày rõ quy mô kết quả.

In [16]:
rules_summary = pd.DataFrame([
    {
        "metric": "Số giao dịch",
        "value": num_transactions
    },
    {
        "metric": "Số frequent itemsets đầu vào",
        "value": len(frequent_itemsets)
    },
    {
        "metric": "Số luật ban đầu",
        "value": len(rules_enriched)
    },
    {
        "metric": "Số luật sau lọc",
        "value": len(rules_filtered)
    },
    {
        "metric": "Số luật dùng cho gợi ý 1 sản phẩm",
        "value": len(recommendation_lookup)
    },
    {
        "metric": "Số sản phẩm đầu vào có thể gợi ý",
        "value": recommendation_lookup["input_product_name"].nunique()
    },
    {
        "metric": "Support tối thiểu khi lọc",
        "value": min_support_rule
    },
    {
        "metric": "Confidence tối thiểu khi lọc",
        "value": min_confidence_rule
    },
    {
        "metric": "Lift tối thiểu khi lọc",
        "value": min_lift_rule
    },
    {
        "metric": "Độ dài luật tối đa",
        "value": max_rule_len
    }
])

display(rules_summary)

,metric,value
0,Số giao dịch,75519.000
1,Số frequent itemsets đầu vào,4269.000
2,Số luật ban đầu,3560.000
3,Số luật sau lọc,896.000
4,Số luật dùng cho gợi ý 1 sản phẩm,450.000
5,Số sản phẩm đầu vào có thể gợi ý,306.000
6,Support tối thiểu khi lọc,0.001
7,Confidence tối thiểu khi lọc,0.200
8,Lift tối thiểu khi lọc,1.200
9,Độ dài luật tối đa,4.000


## 14. Lưu toàn bộ kết quả luật kết hợp

Sau khi hoàn tất quá trình sinh luật, lọc luật và tạo bảng gợi ý, notebook lưu toàn bộ kết quả vào thư mục `results`.

Các file được tạo gồm:

| File | Vai trò |
| --- | --- |
| `association_rules_all.csv` | Toàn bộ luật sinh ra ban đầu |
| `association_rules_filtered.csv` | Các luật đã lọc theo support, confidence và lift |
| `top_rules_by_lift.csv` | Top luật theo lift |
| `top_rules_by_confidence.csv` | Top luật theo confidence |
| `top_rules_by_score.csv` | Top luật theo recommendation_score |
| `top_rules_by_weighted_score.csv` | Top luật theo weighted_recommendation_score |
| `recommendation_lookup.csv` | Bảng dùng trực tiếp cho web demo gợi ý sản phẩm |
| `association_rules_summary.csv` | Bảng tóm tắt kết quả sinh luật |

In [17]:
rules_enriched.to_csv(
    RESULTS_DIR / "association_rules_all.csv",
    index=False,
    encoding="utf-8-sig"
)

rules_filtered.to_csv(
    RESULTS_DIR / "association_rules_filtered.csv",
    index=False,
    encoding="utf-8-sig"
)

top_rules_by_lift.to_csv(
    RESULTS_DIR / "top_rules_by_lift.csv",
    index=False,
    encoding="utf-8-sig"
)

top_rules_by_confidence.to_csv(
    RESULTS_DIR / "top_rules_by_confidence.csv",
    index=False,
    encoding="utf-8-sig"
)

top_rules_by_score.to_csv(
    RESULTS_DIR / "top_rules_by_score.csv",
    index=False,
    encoding="utf-8-sig"
)

top_rules_by_weighted_score.to_csv(
    RESULTS_DIR / "top_rules_by_weighted_score.csv",
    index=False,
    encoding="utf-8-sig"
)

recommendation_lookup.to_csv(
    RESULTS_DIR / "recommendation_lookup.csv",
    index=False,
    encoding="utf-8-sig"
)

rules_summary.to_csv(
    RESULTS_DIR / "association_rules_summary.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Đã xuất file association_rules_all.csv")
print("Đã xuất file association_rules_filtered.csv")
print("Đã xuất file top_rules_by_lift.csv")
print("Đã xuất file top_rules_by_confidence.csv")
print("Đã xuất file top_rules_by_score.csv")
print("Đã xuất file top_rules_by_weighted_score.csv")
print("Đã xuất file recommendation_lookup.csv")
print("Đã xuất file association_rules_summary.csv")

Đã xuất file association_rules_all.csv
Đã xuất file association_rules_filtered.csv
Đã xuất file top_rules_by_lift.csv
Đã xuất file top_rules_by_confidence.csv
Đã xuất file top_rules_by_score.csv
Đã xuất file top_rules_by_weighted_score.csv
Đã xuất file recommendation_lookup.csv
Đã xuất file association_rules_summary.csv


## 15. Nhận xét kết quả sinh luật kết hợp

Kết quả của notebook cho thấy frequent itemsets đã được chuyển thành các luật kết hợp dạng `A -> B`. Sau đó, các luật được lọc bằng nhiều tiêu chí gồm support, confidence, lift và độ dài luật.

Điểm cải tiến của bước này nằm ở hai phần chính:

Thứ nhất, luật không được chọn chỉ dựa trên một chỉ số đơn lẻ. Việc kết hợp support, confidence và lift giúp loại bỏ các luật quá hiếm hoặc các luật chỉ phản ánh độ phổ biến tự nhiên của sản phẩm.

Thứ hai, notebook xây dựng thêm `weighted_recommendation_score` bằng cách kết hợp confidence, lift và support_count. Chỉ số này giúp xếp hạng các luật gợi ý theo hướng cân bằng hơn giữa khả năng mua kèm, mức độ liên hệ và số lần xuất hiện thực tế.

File `recommendation_lookup.csv` là đầu ra quan trọng cho web demo. Ứng dụng sẽ đọc file này để gợi ý sản phẩm mua kèm khi người dùng chọn một sản phẩm đầu vào.

In [18]:
print("Nhận xét tổng quan:")
print("Frequent itemsets đã được chuyển thành association rules dạng A -> B.")
print("Các luật được lọc bằng support, confidence, lift và độ dài luật.")
print("recommendation_score được tính bằng confidence * lift.")
print("weighted_recommendation_score được tính bằng confidence * lift * log(1 + support_count).")
print("File recommendation_lookup.csv đã sẵn sàng để dùng cho web demo gợi ý sản phẩm.")

Nhận xét tổng quan:
Frequent itemsets đã được chuyển thành association rules dạng A -> B.
Các luật được lọc bằng support, confidence, lift và độ dài luật.
recommendation_score được tính bằng confidence * lift.
weighted_recommendation_score được tính bằng confidence * lift * log(1 + support_count).
File recommendation_lookup.csv đã sẵn sàng để dùng cho web demo gợi ý sản phẩm.


## Tổng kết bước sinh association rules

Notebook này đã hoàn thành bước sinh luật kết hợp từ frequent itemsets.

Các kết quả chính gồm:

- Sinh các luật kết hợp dạng `A -> B`.
- Bổ sung tên sản phẩm, nhóm hàng và quầy hàng cho từng luật.
- Đánh giá luật bằng support, confidence, lift, leverage và conviction.
- Lọc luật bằng nhiều tiêu chí để giữ lại các luật có chất lượng tốt.
- Bổ sung `recommendation_score` và `weighted_recommendation_score` như một cải tiến phương pháp cho bài toán gợi ý sản phẩm.
- Tạo `recommendation_lookup.csv` để phục vụ web demo.
- Lưu toàn bộ file kết quả vào thư mục `results`.

Bước tiếp theo là phân tích mở rộng dựa trên dữ liệu có nhãn, gồm phân tích theo department, aisle, reordered và thời gian mua hàng.